# 🎨 Clustering — Menemukan Kelompok Tersembunyi dalam Data

**Modul 1: Machine Learning Fundamentals** | Notebook 8 dari 9

---

## 🎯 Apa yang Akan Kita Pelajari?

1. Apa itu **Unsupervised Learning** dan bedanya dengan Supervised Learning
2. **K-Means Clustering** — mengelompokkan data ke K kelompok
3. Cara menentukan **jumlah cluster optimal** (Elbow & Silhouette)
4. **Hierarchical Clustering** dan cara membaca **dendrogram**
5. **DBSCAN** — clustering berdasarkan kepadatan *(bonus)*

### ⏱️ Estimasi Waktu: ~90 menit + diskusi

---

## 💡 Supervised vs Unsupervised Learning

Sejauh ini, semua model yang kita pelajari adalah **Supervised Learning** — ada "jawaban benar" (label) yang kita coba tebak.

| Supervised Learning | Unsupervised Learning |
|---|---|
| Ada label (ya/tidak, harga, dll) | **Tidak ada label** |
| Model belajar dari contoh jawaban | Model menemukan pola sendiri |
| Contoh: prediksi harga, klasifikasi spam | Contoh: **mengelompokkan pelanggan** |

### Analogi Clustering

Bayangkan kamu adalah guru baru yang belum kenal murid-muridnya. Kamu diminta **membagi kelas menjadi beberapa kelompok** berdasarkan kesamaan mereka — tanpa tahu siapa yang berteman dengan siapa.

Kamu mungkin akan melihat: siapa yang duduk berdekatan? Siapa yang pakai seragam mirip? Siapa yang bawa bekal serupa? → Dari situ kamu membentuk **kelompok-kelompok** secara alami.

Itulah **clustering** — mengelompokkan data berdasarkan **kemiripan**, tanpa dikasih tahu jawabannya terlebih dahulu.

---

## 1️⃣ Persiapan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('✅ Library berhasil di-import!')

---

## 2️⃣ Memuat Data

### 📋 Tentang Dataset

Kita menggunakan dataset **Wholesale Customers** — data pembelian tahunan dari 440 pelanggan sebuah distributor.

| Kolom | Keterangan |
|-------|------------|
| `Channel` | Jenis pelanggan (1=Hotel/Restoran, 2=Retail) |
| `Region` | Wilayah (1, 2, 3) |
| `Fresh` | Pembelian produk segar (tahunan) |
| `Milk` | Pembelian susu |
| `Grocery` | Pembelian bahan makanan |
| `Frozen` | Pembelian produk beku |
| `Detergents_Paper` | Pembelian deterjen & kertas |
| `Delicassen` | Pembelian makanan jadi |

**Pertanyaan:** Bisakah kita menemukan **segmen pelanggan** yang berbeda berdasarkan pola belanja mereka?

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
data = pd.read_csv('Wholesale customers data.csv')
print(f'Dataset: {data.shape[0]} pelanggan, {data.shape[1]} kolom')
data.head()

In [ ]:
# Statistik dasar — perhatikan range yang sangat berbeda antar kolom
data.describe().round(0)

### EDA: Pola Pembelian Pelanggan

In [ ]:
# Heatmap korelasi — fitur mana yang berkaitan?
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Korelasi
produk_cols = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
corr = data[produk_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=axes[0])
axes[0].set_title('Korelasi Antar Produk', fontsize=13, fontweight='bold')

# Distribusi per produk
data[produk_cols].mean().sort_values().plot(kind='barh', ax=axes[1], color='#42A5F5')
axes[1].set_xlabel('Rata-rata Pembelian (tahunan)', fontsize=11)
axes[1].set_title('Rata-rata Pembelian per Kategori', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print('💡 Grocery dan Detergents_Paper sangat berkorelasi (0.92)')
print('   → Pelanggan yang banyak beli groceries biasanya juga beli deterjen')

### Scaling dan PCA

Karena data punya **8 dimensi** (kolom), kita tidak bisa langsung menggambarnya di plot 2D. Kita gunakan **PCA (Principal Component Analysis)** untuk mereduksi ke 2 dimensi yang tetap menangkap pola utama.

In [ ]:
# Scaling — WAJIB untuk clustering!
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

# PCA untuk visualisasi 2D
pca = PCA(n_components=2)
data_2d = pca.fit_transform(data_scaled)

print(f'Variasi data yang tertangkap oleh 2 komponen PCA: {pca.explained_variance_ratio_.sum()*100:.1f}%')

plt.figure(figsize=(8, 6))
plt.scatter(data_2d[:, 0], data_2d[:, 1], alpha=0.5, edgecolors='white', s=50, c='#666666')
plt.xlabel('Komponen 1', fontsize=11)
plt.ylabel('Komponen 2', fontsize=11)
plt.title('Data Pelanggan dalam 2D (PCA)', fontsize=14, fontweight='bold')
plt.show()

print('💡 Tanpa clustering, data terlihat seperti gumpalan tunggal.')
print('   Mari kita lihat apakah ada kelompok-kelompok tersembunyi!')

---

## 3️⃣ K-Means Clustering

### Cara Kerja K-Means:

1. **Pilih K** titik acak sebagai pusat cluster awal (**centroid**)
2. **Assign** setiap data ke centroid terdekat
3. **Pindahkan** centroid ke rata-rata anggotanya
4. **Ulangi** langkah 2-3 sampai stabil

**Analogi:** Seperti menaruh 3 magnet di atas meja yang penuh paku payung. Paku-paku akan menempel ke magnet terdekat. Lalu kamu pindahkan magnet ke tengah-tengah kelompok paku. Ulangi sampai tidak ada paku yang berpindah.

### Tapi berapa K yang tepat?

Kita gunakan dua metode:
1. **Elbow Method** — cari titik "siku" pada grafik inertia
2. **Silhouette Score** — skor kualitas cluster (makin tinggi makin baik)

In [ ]:
# Elbow Method + Silhouette Score
k_range = range(2, 11)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(data_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(data_scaled, km.labels_))

# Plot keduanya
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow
axes[0].plot(list(k_range), inertias, 'o-', color='#2196F3', linewidth=2, markersize=8)
axes[0].set_xlabel('Jumlah Cluster (K)', fontsize=11)
axes[0].set_ylabel('Inertia (SSE)', fontsize=11)
axes[0].set_title('Elbow Method', fontsize=13, fontweight='bold')
axes[0].set_xticks(list(k_range))

# Silhouette
best_k = list(k_range)[np.argmax(silhouettes)]
colors_sil = ['#66BB6A' if k == best_k else '#42A5F5' for k in k_range]
axes[1].bar(list(k_range), silhouettes, color=colors_sil)
axes[1].set_xlabel('Jumlah Cluster (K)', fontsize=11)
axes[1].set_ylabel('Silhouette Score', fontsize=11)
axes[1].set_title('Silhouette Score (makin tinggi = makin baik)', fontsize=13, fontweight='bold')
axes[1].set_xticks(list(k_range))

plt.tight_layout()
plt.show()

print(f'🏆 Silhouette Score tertinggi: K={best_k} (skor={max(silhouettes):.3f})')
print(f'\n💡 Cara baca Elbow: cari titik di mana garis mulai "melengkung" (seperti siku tangan)')
print(f'   Cara baca Silhouette: pilih K dengan skor tertinggi')

In [ ]:
# Jalankan K-Means dengan K optimal
km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = km_best.fit_predict(data_scaled)

print(f'📊 K-Means dengan K={best_k}')
print(f'Jumlah pelanggan per cluster:')
for i in range(best_k):
    count = (cluster_labels == i).sum()
    print(f'  Cluster {i}: {count} pelanggan ({count/len(data)*100:.1f}%)')

In [ ]:
# Visualisasi cluster di 2D (PCA)
plt.figure(figsize=(10, 7))

cmap = plt.cm.Set2
for i in range(best_k):
    mask = cluster_labels == i
    plt.scatter(data_2d[mask, 0], data_2d[mask, 1], label=f'Cluster {i}',
                alpha=0.6, edgecolors='white', s=60, c=[cmap(i)])

# Tampilkan centroid
centroids_2d = pca.transform(km_best.cluster_centers_)
plt.scatter(centroids_2d[:, 0], centroids_2d[:, 1],
            marker='X', s=200, c='red', edgecolors='black', linewidths=2,
            label='Centroid', zorder=5)

plt.xlabel('Komponen 1 (PCA)', fontsize=11)
plt.ylabel('Komponen 2 (PCA)', fontsize=11)
plt.title(f'Hasil K-Means Clustering (K={best_k})', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print('❌ Tanda X merah = centroid (pusat) setiap cluster')

### Profil Setiap Cluster — Siapa Mereka?

In [ ]:
# Analisis profil cluster
data_with_cluster = data.copy()
data_with_cluster['Cluster'] = cluster_labels

# Rata-rata pembelian per cluster
profil = data_with_cluster.groupby('Cluster')[produk_cols].mean().round(0)

print('📊 Profil Rata-rata Pembelian per Cluster:')
print(profil.to_string())

# Heatmap profil
plt.figure(figsize=(10, 4))
# Normalisasi per kolom untuk perbandingan yang adil
profil_norm = profil.div(profil.max(axis=0), axis=1)
sns.heatmap(profil_norm, annot=profil.values.astype(int), fmt='d',
            cmap='YlOrRd', linewidths=1, cbar_kws={'label': 'Relatif terhadap max'})
plt.title('Profil Belanja Setiap Cluster', fontsize=14, fontweight='bold')
plt.ylabel('Cluster', fontsize=11)
plt.tight_layout()
plt.show()

print('\n💡 Warna GELAP = pembelian tinggi, TERANG = rendah')
print('   Dari sini kita bisa memberi "nama" setiap cluster sesuai polanya.')

---

## 4️⃣ Hierarchical Clustering

Berbeda dengan K-Means yang harus menentukan K di awal, **Hierarchical Clustering** membuat **pohon penggabungan** (dendrogram) yang menunjukkan urutan pengelompokan dari bawah ke atas.

### Cara Kerja:
1. Mulai: setiap data adalah cluster sendiri
2. Gabungkan 2 cluster **terdekat**
3. Ulangi sampai semua jadi 1 cluster

**Analogi:** Seperti turnamen sepak bola yang dimulai dari babak penyisihan. Tim-tim bermain, pemenang digabung, sampai akhirnya tinggal 1 juara. Dendrogram menunjukkan "bracket" turnamen ini.

### Metode Linkage:
| Metode | Cara Mengukur Jarak Antar Cluster |
|--------|-----------------------------------|
| **Single** | Jarak titik terdekat (minimum) |
| **Average** | Rata-rata semua jarak |
| **Complete** | Jarak titik terjauh (maksimum) |

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

# Bandingkan 3 metode linkage
methods = ['single', 'average', 'complete']
fig, axes = plt.subplots(3, 1, figsize=(16, 14))

for i, method in enumerate(methods):
    z = linkage(data_scaled, metric='euclidean', method=method)
    dendrogram(z, ax=axes[i], truncate_mode='lastp', p=30,
               leaf_rotation=90, leaf_font_size=8)
    axes[i].set_title(f'Dendrogram — Linkage: {method.upper()}', fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Jarak', fontsize=10)
    axes[i].axhline(y=10, color='red', linestyle='--', alpha=0.5, label='Garis potong contoh')
    axes[i].legend(fontsize=9)

plt.tight_layout()
plt.show()

print('💡 Cara membaca dendrogram:')
print('  - Sumbu Y = jarak antar cluster yang digabung')
print('  - Garis vertikal panjang = cluster yang sangat berbeda')
print('  - Potong di ketinggian tertentu → jumlah cluster = jumlah garis yang terpotong')

In [ ]:
# Jalankan Hierarchical Clustering
n_clusters_hier = best_k  # Gunakan jumlah cluster yang sama untuk perbandingan
hier = AgglomerativeClustering(n_clusters=n_clusters_hier, linkage='average')
hier_labels = hier.fit_predict(data_scaled)

# Visualisasi perbandingan K-Means vs Hierarchical
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, labels, title in [
    (axes[0], cluster_labels, f'K-Means (K={best_k})'),
    (axes[1], hier_labels, f'Hierarchical (K={n_clusters_hier})')
]:
    for i in range(n_clusters_hier):
        mask = labels == i
        ax.scatter(data_2d[mask, 0], data_2d[mask, 1],
                   label=f'Cluster {i}', alpha=0.6, edgecolors='white', s=50)
    sil = silhouette_score(data_scaled, labels)
    ax.set_title(f'{title}\nSilhouette={sil:.3f}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Komponen 1 (PCA)', fontsize=10)
    ax.set_ylabel('Komponen 2 (PCA)', fontsize=10)
    ax.legend(fontsize=9)

plt.suptitle('K-Means vs Hierarchical Clustering', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 🌟 BONUS: DBSCAN — Clustering Berdasarkan Kepadatan

**DBSCAN** (Density-Based Spatial Clustering) berbeda dari K-Means:
- **Tidak perlu menentukan K** di awal
- Bisa menemukan cluster dengan **bentuk apapun** (tidak harus bulat)
- Bisa mendeteksi **outlier** (data yang tidak masuk kelompok manapun)

**Analogi:** Bayangkan kamu melihat kerumunan orang dari atas gedung. DBSCAN menemukan kelompok berdasarkan "siapa yang berdiri berdekatan" — orang yang sendirian jauh dari kerumunan dianggap outlier.

### Parameter:
- `eps` = radius pencarian (seberapa dekat harus berdekatan?)
- `min_samples` = minimum anggota untuk dianggap cluster

In [ ]:
# DBSCAN
dbs = DBSCAN(eps=1.5, min_samples=5)
dbs_labels = dbs.fit_predict(data_scaled)

n_clusters_dbs = len(set(dbs_labels)) - (1 if -1 in dbs_labels else 0)
n_noise = (dbs_labels == -1).sum()

print(f'📊 DBSCAN menemukan:')
print(f'  Cluster: {n_clusters_dbs}')
print(f'  Outlier: {n_noise} data ({n_noise/len(data)*100:.1f}%)')

# Visualisasi
plt.figure(figsize=(10, 7))

# Plot cluster
unique_labels = sorted(set(dbs_labels))
for label in unique_labels:
    mask = dbs_labels == label
    if label == -1:
        plt.scatter(data_2d[mask, 0], data_2d[mask, 1],
                    c='gray', marker='x', s=30, alpha=0.5, label='Outlier')
    else:
        plt.scatter(data_2d[mask, 0], data_2d[mask, 1],
                    label=f'Cluster {label}', alpha=0.6, edgecolors='white', s=60)

plt.xlabel('Komponen 1 (PCA)', fontsize=11)
plt.ylabel('Komponen 2 (PCA)', fontsize=11)
plt.title('DBSCAN — Clustering + Deteksi Outlier', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('\n💡 DBSCAN secara otomatis mendeteksi outlier (x abu-abu)')
print('   Ini berguna untuk mendeteksi pelanggan dengan pola belanja TIDAK BIASA.')

---

## 📝 Ringkasan: Apa yang Kita Pelajari Hari Ini?

### Konsep Utama

1. **Clustering** = mengelompokkan data tanpa label (unsupervised learning)
2. **K-Means** membagi data ke K kelompok berdasarkan jarak ke centroid
3. **Elbow Method** dan **Silhouette Score** membantu menentukan jumlah cluster optimal
4. **Hierarchical Clustering** membuat dendrogram yang menunjukkan hubungan antar cluster
5. **DBSCAN** menemukan cluster berdasarkan kepadatan dan bisa mendeteksi outlier

### Perbandingan Metode Clustering

| Metode | Perlu tentukan K? | Bentuk cluster | Deteksi outlier? |
|--------|-------------------|---------------|------------------|
| **K-Means** | Ya | Bulat/sferis | Tidak |
| **Hierarchical** | Bisa setelahnya | Fleksibel | Tidak |
| **DBSCAN** | Tidak | Bentuk apapun | Ya |

### Aplikasi Clustering di Dunia Nyata

- 🛒 **Segmentasi pelanggan** — seperti yang kita lakukan hari ini!
- 📰 **Pengelompokan berita** berdasarkan topik
- 🎵 **Rekomendasi musik** — lagu yang "mirip" dikelompokkan
- 🏥 **Segmentasi pasien** berdasarkan gejala

---

### 🔜 Notebook Selanjutnya: Time Series Analysis
Kita akan belajar memprediksi data yang berurutan berdasarkan waktu — seperti memprediksi jumlah penumpang pesawat bulan depan!